# FNN to TVB One-Way Coupling Prototype

This notebook implements a first-stage one-way coupling prototype:

- FNN consumes a synthetic visual movie and produces unit-level activity
- unit activity is collapsed into a single population drive
- the drive is injected into TVB visual regions (`rV1`, `rV2`, `lV1`, `lV2`) as a sampled regional stimulus
- baseline and stimulated TVB runs are compared side by side

This is intentionally a phase-1 prototype. It demonstrates the technical contract for `FNN -> TVB` without yet adding `TVB -> FNN` feedback.


In [ ]:
from pathlib import Path
import sys


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "src" / "fnn").exists() and (candidate / "notebook").exists():
            return candidate
    raise FileNotFoundError("Could not find the project root.")


PROJECT_ROOT = find_project_root()
NOTEBOOK_DIR = PROJECT_ROOT / "notebook"
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from fnn_tvb_notebook_utils import ensure_src_on_sys_path, find_project_root as helper_find_project_root
ensure_src_on_sys_path(PROJECT_ROOT)
assert helper_find_project_root(PROJECT_ROOT) == PROJECT_ROOT

import numpy as np
import matplotlib.pyplot as plt


In [ ]:
from fnn_tvb_notebook_utils import (
    build_constant_covariates,
    build_random_fnn,
    build_sampled_region_stimulus,
    build_tvb_simulator,
    extract_region_trace,
    frame_times_ms,
    load_pretrained_fnn,
    make_moving_bar_video,
    predict_responses,
    run_tvb,
    set_reproducible_seed,
    summarize_population_drive,
    unpack_first_monitor,
)

set_reproducible_seed(7)

OUTPUT_DIR = PROJECT_ROOT / "notebook" / "output" / "fnn_tvb_one_way_coupling"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SESSION = 8
SCAN_IDX = 5
PRETRAINED_DIR = PROJECT_ROOT / "notebook" / "data" / "fnn" / "microns_digital_twin" / "params"
PRETRAINED_READY = (PRETRAINED_DIR / "params_core.pt").exists() and (PRETRAINED_DIR / f"params_{SESSION}_{SCAN_IDX}.pt").exists()
USE_PRETRAINED = PRETRAINED_READY

FRAME_COUNT = 90
FPS = 30.0
TARGET_LABELS = ("rV1", "rV2", "lV1", "lV2")
CONTROL_LABEL = "rFEF"
DRIVE_SCALE = 0.25

print("Project root:", PROJECT_ROOT)
print("Output dir:", OUTPUT_DIR)


In [ ]:
frames = make_moving_bar_video(frame_count=FRAME_COUNT)
perspectives, modulations = build_constant_covariates(len(frames))

if USE_PRETRAINED:
    if not PRETRAINED_DIR.exists():
        raise FileNotFoundError(
            f"Pretrained directory not found: {PRETRAINED_DIR}. "
            "Download it first or set USE_PRETRAINED = False."
        )
    model, unit_ids = load_pretrained_fnn(
        params_dir=PRETRAINED_DIR,
        session=SESSION,
        scan_idx=SCAN_IDX,
        cuda=False,
    )
    model_mode = f"pretrained MICrONS digital twin ({SESSION}-{SCAN_IDX})"
else:
    model = build_random_fnn(units=64, randomize=True)
    unit_ids = None
    model_mode = "randomized FNN skeleton"

responses = predict_responses(model, frames, perspectives, modulations)
drive = summarize_population_drive(responses, method="mean", normalize=True)
drive_times_ms = frame_times_ms(len(frames), fps=FPS)
simulation_length_ms = float(drive_times_ms[-1] + 100.0)

print("Model mode:", model_mode)
print("responses shape:", responses.shape)
print("drive min/max:", float(drive.min()), float(drive.max()))
print("simulation_length_ms:", simulation_length_ms)


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
axes[0].plot(drive_times_ms, drive, color="tab:blue", lw=2)
axes[0].set_ylabel("Normalized drive")
axes[0].set_title("FNN population drive")

im = axes[1].imshow(
    responses[:, :16].T,
    aspect="auto",
    origin="lower",
    extent=[drive_times_ms[0], drive_times_ms[-1], 0, min(16, responses.shape[1])],
    cmap="viridis",
)
axes[1].set_xlabel("Time (ms)")
axes[1].set_ylabel("Unit index")
axes[1].set_title("Example unit responses")
fig.colorbar(im, ax=axes[1], label="Predicted activity")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "fnn_drive_summary.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
baseline = build_tvb_simulator(simulation_length_ms=simulation_length_ms)
stimulus = build_sampled_region_stimulus(
    connectivity=baseline.connectivity,
    sample_times_ms=drive_times_ms,
    sample_values=DRIVE_SCALE * drive,
    target_labels=TARGET_LABELS,
    amplitude=1.0,
)
stimulated = build_tvb_simulator(simulation_length_ms=simulation_length_ms, stimulus=stimulus)

baseline_results = run_tvb(baseline, simulation_length_ms=simulation_length_ms)
stimulated_results = run_tvb(stimulated, simulation_length_ms=simulation_length_ms)

baseline_times, baseline_values = unpack_first_monitor(baseline_results)
stimulated_times, stimulated_values = unpack_first_monitor(stimulated_results)


In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=False)
axes[0].plot(drive_times_ms, DRIVE_SCALE * drive, color="tab:blue", lw=2)
axes[0].set_title("Injected FNN-derived regional drive")
axes[0].set_ylabel("Drive amplitude")

for label in TARGET_LABELS:
    _, baseline_trace = extract_region_trace(baseline_times, baseline_values, baseline, region_label=label)
    _, stimulated_trace = extract_region_trace(stimulated_times, stimulated_values, stimulated, region_label=label)
    axes[1].plot(baseline_times, baseline_trace, ls="--", alpha=0.8, label=f"{label} baseline")
    axes[1].plot(stimulated_times, stimulated_trace, lw=1.5, label=f"{label} stimulated")
axes[1].set_title("TVB visual-region response")
axes[1].set_ylabel("State variable 0")
axes[1].legend(ncol=2, fontsize=8)

_, baseline_control = extract_region_trace(baseline_times, baseline_values, baseline, region_label=CONTROL_LABEL)
_, stimulated_control = extract_region_trace(stimulated_times, stimulated_values, stimulated, region_label=CONTROL_LABEL)
axes[2].plot(baseline_times, baseline_control, ls="--", label=f"{CONTROL_LABEL} baseline")
axes[2].plot(stimulated_times, stimulated_control, lw=1.5, label=f"{CONTROL_LABEL} stimulated")
axes[2].set_title("TVB control-region response")
axes[2].set_xlabel("Time (ms)")
axes[2].set_ylabel("State variable 0")
axes[2].legend()

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "fnn_tvb_one_way_coupling.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
summary = {}
for label in [*TARGET_LABELS, CONTROL_LABEL]:
    _, baseline_trace = extract_region_trace(baseline_times, baseline_values, baseline, region_label=label)
    _, stimulated_trace = extract_region_trace(stimulated_times, stimulated_values, stimulated, region_label=label)
    summary[label] = {
        "baseline_mean": float(np.mean(baseline_trace)),
        "stimulated_mean": float(np.mean(stimulated_trace)),
        "mean_delta": float(np.mean(stimulated_trace - baseline_trace)),
    }

summary


In [ ]:
import json

summary_path = OUTPUT_DIR / "summary.json"
summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")
np.savez(
    OUTPUT_DIR / "fnn_tvb_one_way_data.npz",
    drive_times_ms=drive_times_ms,
    drive=drive,
    baseline_times=baseline_times,
    baseline_values=baseline_values,
    stimulated_times=stimulated_times,
    stimulated_values=stimulated_values,
)

print("Saved summary to:", summary_path)
print("Saved arrays to:", OUTPUT_DIR / "fnn_tvb_one_way_data.npz")


## Interpretation

This notebook is the first implementation step of the proposal.

It does not yet replace TVB nodes with FNN internals. Instead it demonstrates a clean one-way contract:
- FNN produces a time series drive from visual input
- TVB consumes that drive at selected visual nodes

The next step, if we want stronger biological meaning, is to replace the randomized FNN skeleton with pretrained scan-specific weights and then define a more principled unit-to-region aggregation rule.
